# Tomahawk Rev2.4 integrated screening — fail-closed v1.5

This notebook runs the corrected Rev2.4 provisional inputs through a fixed-boundary axisymmetric Grad-Shafranov screen, D-T power/confinement balance, coolant conservation screen, and reduced uncertainty envelope. Attach the matching v1.5 Python runner and corrected data ZIP as Kaggle inputs, or attach the rebuilt combined package ZIP. The bootstrap verifies the exact runner version, input manifest, SHA-256 hashes, the 2.70 s active tau-E target, the separate 1.36 s / 63% reference condition, and the 50% full-incident secondary heat routing before any model result is produced. It rejects stale or ambiguous bundles with INPUT_BLOCKED. Results remain preliminary and do not claim 3-D MHD, OpenMC, CFD/CHT, CHF, ignition, or hardware validation.

In [ ]:
import ast
import hashlib
import importlib.util
import os
from pathlib import Path
import zipfile

EXPECTED_RUNNER_VERSION = "TOMAHAWK-REV2.4-INTEGRATED-SCREEN-1.5"
SCRIPT_NAME = "Tomahawk_Rev2_4_Integrated_Screening.py"

KAGGLE_INPUT_ROOT = Path(
    os.environ.get("TOMAHAWK_KAGGLE_INPUT_ROOT", "/kaggle/input")
).expanduser().resolve()
KAGGLE_WORKING_ROOT = Path(
    os.environ.get("TOMAHAWK_KAGGLE_WORKING_ROOT", "/kaggle/working")
).expanduser().resolve()
OUT = Path(
    os.environ.get(
        "TOMAHAWK_OUTPUT_ROOT",
        str(KAGGLE_WORKING_ROOT / "tomahawk_rev24_screen"),
    )
).expanduser().resolve()
KAGGLE_WORKING_ROOT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)


def input_blocked(reason):
    raise RuntimeError(f"INPUT_BLOCKED: {reason}")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def unique_existing(paths):
    result = []
    for path in paths:
        path = Path(path).expanduser().resolve()
        if path.exists() and path not in result:
            result.append(path)
    return result


def safe_extract(archive, target):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(archive) as handle:
            for member in handle.infolist():
                destination = (target / member.filename).resolve()
                if destination != target and target not in destination.parents:
                    input_blocked(
                        f"unsafe archive member in {Path(archive).name}: "
                        f"{member.filename}"
                    )
            handle.extractall(target)
    except (OSError, zipfile.BadZipFile) as exc:
        input_blocked(f"cannot extract {Path(archive).name}: {exc}")


initial_roots = unique_existing(
    (KAGGLE_INPUT_ROOT, KAGGLE_WORKING_ROOT, Path.cwd())
)
archive_names = {
    "Tomahawk_Rev2_4_Integrated_Screening_Package.zip",
    "tomahawk_rev24_confinement_handoff.zip",
    "tomahawk_rev24_data_inputs.zip",
}
archives = []
for root in initial_roots:
    if not root.is_dir():
        continue
    try:
        for archive in root.rglob("*.zip"):
            lower_name = archive.name.lower()
            if archive.name in archive_names or (
                "tomahawk" in lower_name
                and any(
                    token in lower_name
                    for token in ("package", "handoff", "data_inputs")
                )
            ):
                archives.append(archive.resolve())
    except OSError:
        continue

extracted_roots = []
for archive in sorted(set(archives), key=str):
    target = (
        KAGGLE_WORKING_ROOT
        / "tomahawk_rev24_inputs"
        / f"{archive.stem}_{sha256(archive)[:12]}"
    )
    safe_extract(archive, target)
    extracted_roots.append(target.resolve())

roots = unique_existing(
    (KAGGLE_INPUT_ROOT, *extracted_roots, KAGGLE_WORKING_ROOT, Path.cwd())
)


def declared_runner_version(path):
    try:
        tree = ast.parse(Path(path).read_text(encoding="utf-8"))
    except (OSError, SyntaxError, UnicodeError):
        return None
    for node in tree.body:
        if not isinstance(node, ast.Assign):
            continue
        if not any(isinstance(target, ast.Name) and target.id == "VERSION"
                   for target in node.targets):
            continue
        if isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
            return node.value.value
    return None


def path_priority(path):
    path = Path(path).resolve()
    for index, root in enumerate(roots):
        try:
            path.relative_to(root)
            return index, len(str(path)), str(path)
        except ValueError:
            continue
    return len(roots), len(str(path)), str(path)


explicit_runner = os.environ.get("TOMAHAWK_RUNNER_PATH")
if explicit_runner:
    script_candidates = [Path(explicit_runner).expanduser().resolve()]
else:
    script_candidates = []
    for root in roots:
        if not root.is_dir():
            continue
        try:
            script_candidates.extend(root.rglob(SCRIPT_NAME))
        except OSError:
            continue
script_candidates = sorted(
    {path.resolve() for path in script_candidates if path.is_file()},
    key=path_priority,
)
if not script_candidates:
    input_blocked(
        f"no {SCRIPT_NAME} found; attach the v1.5 runner or corrected package ZIP"
    )

version_map = {
    path: declared_runner_version(path) for path in script_candidates
}
valid_scripts = [
    path for path, version in version_map.items()
    if version == EXPECTED_RUNNER_VERSION
]
if not valid_scripts:
    found = ", ".join(
        f"{path.name}={version or 'unreadable'}"
        for path, version in version_map.items()
    )
    input_blocked(
        f"runner version mismatch; expected {EXPECTED_RUNNER_VERSION}; found {found}"
    )
runner_hashes = {sha256(path) for path in valid_scripts}
if len(runner_hashes) != 1:
    input_blocked(
        "multiple different v1.5 runners were found; set TOMAHAWK_RUNNER_PATH"
    )
SCRIPT = sorted(valid_scripts, key=path_priority)[0]

spec = importlib.util.spec_from_file_location("tomahawk_rev24_screen", SCRIPT)
if spec is None or spec.loader is None:
    input_blocked(f"cannot import runner: {SCRIPT}")
screen = importlib.util.module_from_spec(spec)
spec.loader.exec_module(screen)
if getattr(screen, "VERSION", None) != EXPECTED_RUNNER_VERSION:
    input_blocked(
        f"imported runner reports {getattr(screen, 'VERSION', None)!r}; "
        f"expected {EXPECTED_RUNNER_VERSION}"
    )


def is_input_root(path):
    path = Path(path)
    return (path / "MHD" / "mhd_profile_sweep.csv").is_file()


explicit_input_root = os.environ.get("TOMAHAWK_INPUT_ROOT")
if explicit_input_root:
    input_candidates = [Path(explicit_input_root).expanduser().resolve()]
else:
    input_candidates = []
    for root in roots:
        if not root.is_dir():
            continue
        if is_input_root(root):
            input_candidates.append(root.resolve())
        try:
            for profile in root.rglob("mhd_profile_sweep.csv"):
                if profile.parent.name == "MHD" and is_input_root(
                    profile.parent.parent
                ):
                    input_candidates.append(profile.parent.parent.resolve())
        except OSError:
            continue
input_candidates = sorted(set(input_candidates), key=path_priority)
if not input_candidates:
    input_blocked(
        "no provisional_inputs directory found; attach the corrected data ZIP"
    )

valid_inputs = []
rejected_inputs = []
for candidate in input_candidates:
    try:
        audit = screen.audit_input_root(candidate)
        valid_inputs.append((candidate, audit))
    except Exception as exc:
        rejected_inputs.append(f"{candidate}: {exc}")
if not valid_inputs:
    detail = "; ".join(rejected_inputs[:4])
    input_blocked(f"no verified Rev2.4 input bundle: {detail}")
fingerprints = {audit["manifest_sha256"] for _, audit in valid_inputs}
if len(fingerprints) != 1:
    input_blocked(
        "multiple different verified input bundles were found; "
        "set TOMAHAWK_INPUT_ROOT"
    )
INPUT_ROOT, INPUT_AUDIT = sorted(
    valid_inputs, key=lambda item: path_priority(item[0])
)[0]

print("Runner:", screen.VERSION)
print("Runner SHA256:", sha256(SCRIPT))
print("Script:", SCRIPT)
print("Inputs:", INPUT_ROOT)
print("Input manifest SHA256:", INPUT_AUDIT["manifest_sha256"])
print("Verified input files:", INPUT_AUDIT["verified_file_count"])
print("Output:", OUT)


## Smoke run

The smoke run verifies runner identity, manifest hashes, required inputs and artifacts, profile closure, the sparse equilibrium solve, coolant conservation, strict JSON writing, and the explicit external-blocker policy.

In [ ]:
smoke = screen.run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "smoke",
    label="smoke",
    cases=int(os.environ.get("TOMAHAWK_SMOKE_CASES", "256")),
    seed=20260905,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
smoke_checks = {
    "runner_version": smoke["runner_version"],
    "input_source_status": smoke["input_source_status"],
    "input_bundle_verified": smoke["gates"]["input_bundle_verified"],
    "overall_status": smoke["overall_status"],
    "GS_residual_pass": smoke["gates"]["GS_residual_pass"],
    "profile_consistency_pass": smoke["gates"]["profile_consistency_pass"],
    "reference_self_heating_reconciliation_pass": smoke["gates"][
        "reference_self_heating_reconciliation_pass"
    ],
    "alpha_self_heating_pass": smoke["gates"]["alpha_self_heating_pass"],
    "tau_E_margin_pass": smoke["gates"]["tau_E_margin_pass"],
    "confinement_feasibility_screen_pass": smoke["gates"][
        "confinement_feasibility_screen_pass"
    ],
    "coolant_screen_pass": smoke["gates"]["coolant_screen_pass"],
}
required_output_names = (
    "manifest.json",
    "equilibrium_metrics.json",
    "confinement_metrics.json",
    "coolant_summary.json",
    "profile_consistency.json",
    "gate_results.json",
    "equilibrium_fields.npz",
    "uncertainty/reduced_uncertainty_summary.json",
    "uncertainty/reduced_uncertainty_cases.csv",
)
missing_smoke_outputs = [
    name for name in required_output_names if not (OUT / "smoke" / name).is_file()
]
print(smoke_checks)
if not (
    smoke_checks["runner_version"] == EXPECTED_RUNNER_VERSION
    and smoke_checks["input_source_status"] == "BUNDLE_INPUTS_VERIFIED"
    and smoke_checks["input_bundle_verified"] is True
    and smoke_checks["overall_status"] == "SCREENING_PASS_WITH_EXTERNAL_BLOCKERS"
    and smoke_checks["GS_residual_pass"] is True
    and smoke_checks["profile_consistency_pass"] is True
    and smoke_checks["reference_self_heating_reconciliation_pass"] is True
    and smoke_checks["alpha_self_heating_pass"] is True
    and smoke_checks["tau_E_margin_pass"] is True
    and smoke_checks["confinement_feasibility_screen_pass"] is True
    and smoke_checks["coolant_screen_pass"] is True
    and not missing_smoke_outputs
):
    raise RuntimeError(
        f"SMOKE_FAIL: checks={smoke_checks}; "
        f"missing_outputs={missing_smoke_outputs}"
    )
print(
    "SMOKE_PASS: verified inputs, numerical screens, coolant screen, and "
    "required artifacts are green; external solver blockers remain explicit."
)


## Full run

The full run repeats the verified solve and evaluates the reduced 65,536-case parameter envelope. These uncertainty cases are scaled screens and do not re-solve Grad-Shafranov for every sample; their pass fraction is not a validation probability.

In [ ]:
full = screen.run_native_screen(
    input_root=INPUT_ROOT,
    output_root=OUT / "full",
    label="full",
    cases=int(os.environ.get("TOMAHAWK_FULL_CASES", "65536")),
    seed=20260906,
    nr=161,
    nz=161,
    max_gs_iters=12000,
)
full_checks = {
    "runner_version": full["runner_version"],
    "input_source_status": full["input_source_status"],
    "input_bundle_verified": full["gates"]["input_bundle_verified"],
    "overall_status": full["overall_status"],
    "confinement_numerical_screen_pass": full["gates"][
        "confinement_numerical_screen_pass"
    ],
    "fusion_GW_module": full["confinement_metrics"]["fusion_GW_module"],
    "alpha_self_heating_ratio": full["confinement_metrics"][
        "alpha_self_heating_ratio"
    ],
    "reference_condition_self_heating_ratio": full["confinement_metrics"][
        "reference_condition_self_heating_ratio"
    ],
    "reference_self_heating_reconciliation_pass": full["gates"][
        "reference_self_heating_reconciliation_pass"
    ],
    "coolant_screen_pass": full["gates"]["coolant_screen_pass"],
    "cases": full["uncertainty_summary"]["cases"],
    "all_gate_pass_percent": full["uncertainty_summary"][
        "all_gate_pass_percent"
    ],
}
missing_full_outputs = [
    name for name in required_output_names if not (OUT / "full" / name).is_file()
]
print(full_checks)
if not (
    full_checks["runner_version"] == EXPECTED_RUNNER_VERSION
    and full_checks["input_source_status"] == "BUNDLE_INPUTS_VERIFIED"
    and full_checks["input_bundle_verified"] is True
    and full_checks["overall_status"] == "SCREENING_PASS_WITH_EXTERNAL_BLOCKERS"
    and full_checks["confinement_numerical_screen_pass"] is True
    and full_checks["reference_self_heating_reconciliation_pass"] is True
    and full_checks["coolant_screen_pass"] is True
    and not missing_full_outputs
):
    raise RuntimeError(
        f"FULL_SCREEN_FAIL: checks={full_checks}; "
        f"missing_outputs={missing_full_outputs}"
    )
print(
    f"FULL_SCREEN_PASS: the {full_checks['cases']:,}-case reduced envelope "
    "completed with verified artifacts; external solver blockers remain explicit."
)


In [ ]:
import json
for name in ("equilibrium_metrics.json", "confinement_metrics.json", "coolant_summary.json", "gate_results.json"):
    path = OUT / "full" / name
    print("\n", name)
    print(json.dumps(json.loads(path.read_text()), indent=2)[:6000])

bundle = screen.zip_directory(OUT)
print("\nBundle:", bundle)
print("Plots:", sorted(str(path) for path in (OUT / "full").glob("*.png")))

## Interpretation

A fixed-boundary numerical pass confirms only that the declared finite-difference equation was solved to the declared residual. The active tau-E screen is 2.70 s; the recorded 1.36 s / 63% point is retained as a separately reconciled reference condition. The full-incident heat row routes 50% to secondary systems and keeps 50% cassette-local; secondary-system capacity and the deposition split remain unqualified. The uncertainty pass fraction describes only the declared reduced scaling envelope. Free-boundary 3-D MHD, nonlinear stability, orbit-resolved alpha transport, nuclear transport, qualified CFD/CHT, CHF/DNBR, and hardware validation remain external blockers. Treat the generated bundle as a reproducible screening artifact, not an engineering validation package.